# 📊 Social Media Trend Analysis — NLP Enhanced (v2)

**Improvements in v2:**
- 🆕 Platform-level engagement analysis (Twitter / Instagram / LinkedIn / Reddit / TikTok)
- 🆕 Engagement-rate metric (normalised by follower count)
- 🆕 Location-based trend mapping
- 🆕 User-influence scoring
- 🆕 Shares & comments fields in addition to likes
- 🆕 Word cloud visualisation
- 😊 Sentiment Analysis (TextBlob)
- 📊 TF-IDF Keyword Extraction
- 📝 Bigram Phrase Detection
- 🧠 LDA Topic Modeling

---

## 📦 Install Dependencies

In [ ]:
%pip install textblob wordcloud -q
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

## 📥 Load & Inspect Data

In [ ]:
# ─── Update DATA_PATH to your Databricks Volume or local path ───
DATA_PATH = "/Volumes/workspace/default/my_files/social_media_posts.csv"

df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)

print(f"📈 Loaded {df.count():,} social media posts")
print(f"📋 Schema:")
df.printSchema()
display(df.limit(5))

## 🔍 Data Quality Check

In [ ]:
from pyspark.sql.functions import col, count, when, isnan

print("🔎 Null/missing value counts per column:")
null_counts = df.select(
    [count(when(col(c).isNull() | isnan(c), c)).alias(c)
     for c in df.columns]
)
display(null_counts)

print(f"\n📊 Platform distribution:")
display(df.groupBy('platform').count().orderBy(col('count').desc()))

## 🧹 Data Cleaning & Preprocessing

In [ ]:
from pyspark.sql.functions import lower, regexp_replace

clean_df = (
    df.dropDuplicates()
      .filter(col("text").isNotNull())
      .filter(col("likes").cast("int").isNotNull())
      .withColumn("clean_text",
                  lower(regexp_replace(col("text"), r"http\S+|[^a-zA-Z# ]", "")))
      .withColumn("likes",          col("likes").cast("int"))
      .withColumn("shares",         col("shares").cast("int"))
      .withColumn("comments",       col("comments").cast("int"))
      .withColumn("follower_count", col("follower_count").cast("long"))
)

original_count = df.count()
clean_count    = clean_df.count()
print(f"✅ Original records : {original_count:,}")
print(f"✅ After cleaning   : {clean_count:,}")
print(f"🗑️  Removed          : {original_count - clean_count:,}")

## 📐 Engagement Rate (normalised by follower count)

In [ ]:
from pyspark.sql.functions import round as spark_round

# engagement_rate = (likes + shares + comments) / follower_count * 100
engmt_df = clean_df.withColumn(
    "engagement_rate",
    spark_round(
        (col("likes") + col("shares") + col("comments"))
        / (col("follower_count") + 1) * 100,
        4
    )
)

from pyspark.sql.functions import avg, max as spark_max

print("📊 AVERAGE ENGAGEMENT RATE BY PLATFORM:")
display(
    engmt_df.groupBy("platform")
            .agg(
                avg("engagement_rate").alias("avg_engagement_rate_%"),
                avg("likes").alias("avg_likes"),
                avg("shares").alias("avg_shares"),
                avg("comments").alias("avg_comments"),
            )
            .orderBy(col("avg_engagement_rate_%").desc())
)

## 😊 Sentiment Analysis (TextBlob NLP)

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType, StringType
from textblob import TextBlob

@udf(returnType=FloatType())
def get_sentiment_polarity(text):
    """Returns sentiment polarity: -1 (negative) to +1 (positive)"""
    if text is None:
        return 0.0
    try:
        return float(TextBlob(text).sentiment.polarity)
    except Exception:
        return 0.0

@udf(returnType=FloatType())
def get_sentiment_subjectivity(text):
    """Returns subjectivity: 0 (objective) to 1 (subjective)"""
    if text is None:
        return 0.0
    try:
        return float(TextBlob(text).sentiment.subjectivity)
    except Exception:
        return 0.0

@udf(returnType=StringType())
def get_sentiment_label(polarity):
    """Classify sentiment as Positive / Negative / Neutral."""
    if polarity is None:
        return "Neutral"
    return "Positive" if polarity > 0.1 else ("Negative" if polarity < -0.1 else "Neutral")

# Apply sentiment to cleaned text
sentiment_df = (
    engmt_df
    .withColumn("sentiment_polarity",     get_sentiment_polarity(col("text")))
    .withColumn("sentiment_subjectivity", get_sentiment_subjectivity(col("text")))
    .withColumn("sentiment_label",        get_sentiment_label(col("sentiment_polarity")))
)

In [ ]:
print("📊 SENTIMENT DISTRIBUTION:")
display(
    sentiment_df.groupBy("sentiment_label")
                .count()
                .orderBy(col("count").desc())
)

In [ ]:
print("📈 SENTIMENT STATS:")
from pyspark.sql.functions import min as spark_min
display(
    sentiment_df.agg(
        avg("sentiment_polarity").alias("avg_polarity"),
        avg("sentiment_subjectivity").alias("avg_subjectivity"),
        spark_max("sentiment_polarity").alias("max_polarity"),
        spark_min("sentiment_polarity").alias("min_polarity"),
    )
)

In [ ]:
# Sentiment breakdown per platform
print("📱 SENTIMENT BY PLATFORM:")
display(
    sentiment_df.groupBy("platform", "sentiment_label")
                .count()
                .orderBy("platform", col("count").desc())
)

## 🏷️ Hashtag Analysis

In [ ]:
from pyspark.sql.functions import explode, split

hashtags_df = (
    sentiment_df
    .withColumn("hashtag", explode(split(col("hashtags"), "\\|")))
    .filter(col("hashtag") != "")
)

top_hashtags = (
    hashtags_df.groupBy("hashtag")
               .agg(
                   count("*").alias("count"),
                   avg("sentiment_polarity").alias("avg_sentiment"),
                   avg("likes").alias("avg_likes"),
               )
               .orderBy(col("count").desc())
)

total_hashtags  = hashtags_df.count()
unique_hashtags = top_hashtags.count()
print(f"📊 Total mentions: {total_hashtags:,} | Unique: {unique_hashtags:,}")
print("\n🔥 TOP 15 HASHTAGS WITH SENTIMENT & AVG LIKES:")
display(top_hashtags.limit(15))

## 🌍 Location-based Trend Analysis

In [ ]:
print("📍 TOP LOCATIONS BY POST VOLUME:")
display(
    sentiment_df.groupBy("location")
                .agg(
                    count("*").alias("post_count"),
                    avg("likes").alias("avg_likes"),
                    avg("sentiment_polarity").alias("avg_sentiment"),
                )
                .orderBy(col("post_count").desc())
                .limit(15)
)

In [ ]:
# Top category per location
from pyspark.sql.functions import first
from pyspark.sql import Window

loc_cat = (
    sentiment_df.groupBy("location", "category")
                .agg(count("*").alias("cnt"))
)
w_loc = Window.partitionBy("location").orderBy(col("cnt").desc())

from pyspark.sql.functions import rank
top_cat_per_loc = (
    loc_cat.withColumn("rnk", rank().over(w_loc))
            .filter(col("rnk") == 1)
            .drop("rnk")
            .withColumnRenamed("category", "top_category")
            .withColumnRenamed("cnt", "post_count")
            .orderBy(col("post_count").desc())
)
print("🗺️ DOMINANT CATEGORY PER LOCATION:")
display(top_cat_per_loc.limit(15))

## 🌟 User Influence Scoring

In [ ]:
# influence_score = log(follower_count+1) * avg_engagement_rate
from pyspark.sql.functions import log

user_influence = (
    sentiment_df.groupBy("username", "platform")
                .agg(
                    count("*").alias("total_posts"),
                    avg("follower_count").alias("avg_followers"),
                    avg("engagement_rate").alias("avg_eng_rate"),
                    avg("likes").alias("avg_likes"),
                    avg("sentiment_polarity").alias("avg_sentiment"),
                )
                .withColumn(
                    "influence_score",
                    spark_round(
                        log(col("avg_followers") + 1) * col("avg_eng_rate"),
                        4
                    )
                )
                .orderBy(col("influence_score").desc())
)

print("⭐ TOP INFLUENCERS BY INFLUENCE SCORE:")
display(user_influence.limit(20))

## 📝 Bigram Analysis (Common Two-Word Phrases)

In [ ]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, NGram

tokenizer = Tokenizer(inputCol="clean_text", outputCol="tokens_raw")
tokens_df = tokenizer.transform(sentiment_df)

remover = StopWordsRemover(inputCol="tokens_raw", outputCol="tokens")
tokens_df = remover.transform(tokens_df)

ngram = NGram(n=2, inputCol="tokens", outputCol="bigrams")
bigram_df = ngram.transform(tokens_df)

bigrams_exploded = bigram_df.select(explode(col("bigrams")).alias("bigram"))
bigram_counts = (
    bigrams_exploded.groupBy("bigram")
                    .count()
                    .orderBy(col("count").desc())
)

print("🔤 TOP 15 BIGRAMS (Common Phrases):")
display(bigram_counts.limit(15))

## 📊 TF-IDF Keyword Extraction

In [ ]:
from pyspark.ml.feature import CountVectorizer, IDF

cv = CountVectorizer(inputCol="tokens", outputCol="features", minDF=2, vocabSize=1000)
cv_model = cv.fit(tokens_df)
tf_df = cv_model.transform(tokens_df)

idf = IDF(inputCol="features", outputCol="tfidf_features")
idf_model = idf.fit(tf_df)
tfidf_df = idf_model.transform(tf_df)

vocab       = cv_model.vocabulary
idf_weights = idf_model.idf.toArray()
word_scores = [(vocab[i], float(idf_weights[i])) for i in range(len(vocab))]

print("🔑 CORE KEYWORDS (lowest IDF = appears in most documents):")
for i, (word, score) in enumerate(sorted(word_scores, key=lambda x: x[1])[:15], 1):
    print(f"   {i:2}. {word:<20} (IDF: {score:.2f})")

print("\n🔍 DISTINCTIVE KEYWORDS (highest IDF = most unique terms):")
for i, (word, score) in enumerate(sorted(word_scores, key=lambda x: x[1], reverse=True)[:10], 1):
    print(f"   {i:2}. {word:<20} (IDF: {score:.2f})")

## 🧠 Topic Modeling with LDA

In [ ]:
import numpy as np
from pyspark.ml.clustering import LDA

NUM_TOPICS = 8  # increased from 5 for richer topic coverage
print(f"Training LDA with {NUM_TOPICS} topics ...")

lda = LDA(k=NUM_TOPICS, maxIter=15, optimizer="online")
lda_model = lda.fit(tf_df)

print(f"Log Likelihood : {lda_model.logLikelihood(tf_df):.2f}")
print(f"Perplexity     : {lda_model.logPerplexity(tf_df):.2f}")

In [ ]:
topics = lda_model.describeTopics(maxTermsPerTopic=8).collect()

topic_labels = {}
print("🏷️ DISCOVERED TOPICS:")
print("-" * 60)
for topic in topics:
    topic_id    = topic['topic']
    term_indices = topic['termIndices']
    words = [vocab[i] for i in term_indices if i < len(vocab)]
    top_word = words[0].capitalize() if words else f"Topic{topic_id}"
    topic_labels[topic_id] = f"{top_word}-related"
    print(f"\n📌 Topic {topic_id}: {topic_labels[topic_id]}")
    print(f"   Keywords: {', '.join(words[:6])}")

In [ ]:
from pyspark.sql.types import IntegerType

@udf(returnType=IntegerType())
def get_max_topic_index(v):
    if v is None:
        return None
    return int(np.argmax(v.toArray()))

topic_df = lda_model.transform(tf_df)
topic_df = topic_df.withColumn("topic", get_max_topic_index(col("topicDistribution")))

print("📊 TOPIC DISTRIBUTION:")
display(topic_df.groupBy("topic").count().orderBy(col("count").desc()))

## 📊 Topic × Sentiment × Platform

In [ ]:
topic_sentiment = (
    topic_df.groupBy("topic", "platform")
            .agg(
                count("*").alias("post_count"),
                avg("sentiment_polarity").alias("avg_sentiment"),
                avg("engagement_rate").alias("avg_eng_rate"),
            )
            .orderBy("topic", "platform")
)
print("📈 SENTIMENT & ENGAGEMENT BY TOPIC × PLATFORM:")
display(topic_sentiment)

## 📈 Trend Scoring & Momentum Analysis

In [ ]:
from pyspark.sql.functions import to_date, lag, coalesce, lit, log
from pyspark.sql.window import Window

topic_trend = (
    topic_df
    .withColumn("date", to_date("timestamp"))
    .groupBy("date", "topic")
    .agg(
        count("*").alias("count"),
        avg("sentiment_polarity").alias("daily_sentiment"),
        avg("engagement_rate").alias("daily_eng_rate"),
    )
)

w = Window.partitionBy("topic").orderBy("date")
trend_df = (
    topic_trend
    .withColumn("prev_count", coalesce(lag("count").over(w), lit(0)))
    .withColumn("growth_rate", (col("count") - col("prev_count")) / (col("prev_count") + 1))
)

momentum_window = Window.partitionBy("topic").orderBy("date").rowsBetween(-3, 0)
trend_df = trend_df.withColumn("momentum", avg("growth_rate").over(momentum_window))

# Composite trend score: volume + growth + momentum + sentiment + engagement
trend_df = trend_df.withColumn(
    "final_trend_score",
    (log(col("count") + 1)      * 0.25)
    + (col("growth_rate")        * 0.25)
    + (col("momentum")           * 0.20)
    + (col("daily_sentiment")    * 0.15)
    + (col("daily_eng_rate")     * 0.15)
)

In [ ]:
from pyspark.sql.functions import create_map

mapping_expr = create_map([lit(x) for item in topic_labels.items() for x in item])

final_output = (
    trend_df
    .withColumnRenamed("topic", "topic_id")
    .withColumn("topic_name", mapping_expr.getItem(col("topic_id")))
    .select(
        "date", "topic_id", "topic_name", "count",
        spark_round("growth_rate",       3).alias("growth_rate"),
        spark_round("daily_sentiment",   3).alias("sentiment"),
        spark_round("daily_eng_rate",    4).alias("engagement_rate"),
        spark_round("final_trend_score", 4).alias("trend_score"),
    )
)

print("🔥 TOP TRENDING TOPICS (composite score):")
display(final_output.orderBy(col("trend_score").desc()).limit(20))

## 🚀 Fastest Growing Topics

In [ ]:
from pyspark.sql.functions import sum as spark_sum

fastest_growing = (
    final_output
    .filter(col("growth_rate").isNotNull())
    .groupBy("topic_id", "topic_name")
    .agg(
        avg("growth_rate").alias("avg_growth"),
        avg("sentiment").alias("avg_sentiment"),
        avg("engagement_rate").alias("avg_eng_rate"),
        spark_sum("count").alias("total_posts"),
    )
    .orderBy(col("avg_growth").desc())
)

print("🚀 FASTEST GROWING TOPICS:")
display(fastest_growing)

## 💾 Save Results to Parquet

In [ ]:
# Save enriched data
OUTPUT_BASE = "/tmp/social_media_analysis_output"

(
    sentiment_df
    .select("post_id", "username", "platform", "location",
            "category", "hashtags", "likes", "shares", "comments",
            "follower_count", "engagement_rate",
            "sentiment_polarity", "sentiment_label", "timestamp")
    .write.mode("overwrite")
    .parquet(f"{OUTPUT_BASE}/processed_posts")
)

(
    top_hashtags.limit(100)
    .write.mode("overwrite")
    .parquet(f"{OUTPUT_BASE}/trending_hashtags")
)

(
    final_output
    .write.mode("overwrite")
    .parquet(f"{OUTPUT_BASE}/trend_scores")
)

print(f"✅ Results saved to {OUTPUT_BASE}/")

## 📋 Summary Dashboard

In [ ]:
from pyspark.sql.functions import to_date

date_stats    = sentiment_df.select(
    spark_min(to_date("timestamp")).alias("start_date"),
    spark_max(to_date("timestamp")).alias("end_date"),
).collect()[0]

top_3_hashtags = top_hashtags.limit(3).collect()
top_bigram     = bigram_counts.first()
hottest        = final_output.orderBy(col("trend_score").desc()).first()
top_influencer = user_influence.first()

sep = "=" * 65
print(sep)
print("📊 NLP-ENHANCED SOCIAL MEDIA ANALYSIS — SUMMARY (v2)")
print(sep)

print(f"\n📈 DATA OVERVIEW")
print(f"   • Posts analysed : {clean_count:,}")
print(f"   • Hashtag mentions: {total_hashtags:,} ({unique_hashtags:,} unique)")
print(f"   • LDA Topics      : {NUM_TOPICS}")
print(f"   • Date range      : {date_stats['start_date']} → {date_stats['end_date']}")

print(f"\n🧠 NLP TECHNIQUES APPLIED")
for t in ["TextBlob Sentiment Analysis", "TF-IDF Keyword Extraction",
          "Bigram Phrase Detection", "LDA Topic Modeling",
          "Engagement-Rate Normalisation", "User Influence Scoring",
          "Location-Based Trend Mapping"]:
    print(f"   ✅ {t}")

print(f"\n🏆 KEY FINDINGS")
if top_3_hashtags:
    print(f"   • Top hashtag   : #{top_3_hashtags[0]['hashtag']} ({top_3_hashtags[0]['count']} uses)")
if top_bigram:
    print(f"   • Top phrase    : \"{top_bigram['bigram']}\" ({top_bigram['count']} occurrences)")
if hottest:
    print(f"   • Hottest topic : {hottest['topic_name']} (score {hottest['trend_score']})")
if top_influencer:
    print(f"   • Top influencer: @{top_influencer['username']} on {top_influencer['platform']}")
    print(f"     influence score {top_influencer['influence_score']:.4f}")

print(f"\n{sep}")
print("✅ ANALYSIS COMPLETE!")
print(sep)